# Week 5 -- Function 5: PyTorch NN + GP + SVM Active Filter (4D)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 20
DATA_PATH_X  = '../data/function_5/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_5/initial_outputs.npy'

weekly_log = [
    ([0.209005, 0.838746, 0.859156, 0.882439], 984.4093632633864),  # W1: 984
    ([0.204881, 0.87783, 0.879582, 0.870578], 1192.2995655092311),  # W2: 1192 (+21%)
    ([0.204881, 0.87783, 0.879582, 0.870578], 1192.2995655092311),  # W3: plateau
    ([0.133474, 0.97783, 0.979582, 0.970578], 3531.0551965197214),  # W4: 3531 (HUGE JUMP)
    # Week 5: add result here as ([x...], y_value)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Synced 1 new observation(s).
X: (24, 4) | Y: (24,)
Week 5 | 24 total observations (20 initial + 4 added)


In [2]:
# Cell 3: Load data and inspect -- Function 5: Chemical Yield Optimisation (4D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.6e}')
print(f'  Best X*  = [{best_x_str}]')

Input  shape : (24, 4)
Output shape : (24,)

  All observations (sorted descending by Y)
  [ 1]  X = [0.133474, 0.977830, 0.979582, 0.970578]   Y = +3.531055e+03  <-- best
  [ 2]  X = [0.204881, 0.877830, 0.879582, 0.870578]   Y = +1.192300e+03
  [ 3]  X = [0.204881, 0.877830, 0.879582, 0.870578]   Y = +1.192300e+03
  [ 4]  X = [0.224189, 0.846480, 0.879484, 0.878516]   Y = +1.088860e+03
  [ 5]  X = [0.209005, 0.838746, 0.859156, 0.882439]   Y = +9.844094e+02
  [ 6]  X = [0.119879, 0.862540, 0.643331, 0.849804]   Y = +4.316128e+02
  [ 7]  X = [0.438933, 0.774092, 0.378167, 0.933696]   Y = +3.558068e+02
  [ 8]  X = [0.836478, 0.193610, 0.663893, 0.785649]   Y = +2.583705e+02
  [ 9]  X = [0.463442, 0.630025, 0.107906, 0.957644]   Y = +2.332236e+02
  [10]  X = [0.352356, 0.322242, 0.116979, 0.473113]   Y = +1.095719e+02
  [11]  X = [0.511142, 0.817957, 0.728710, 0.112354]   Y = +7.972913e+01
  [12]  X = [0.683432, 0.118663, 0.829046, 0.567577]   Y = +7.843439e+01
  [13]  X = [0.191447, 0.

In [3]:
# Cell 4: Fit GP
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

Y_fit = np.log(np.abs(Y) + 1e-300)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print(f'  Sanity check at best X* : GP mean={pred_mean[0]:.4f}  actual={actual_log:.4f}  std={pred_std[0]:.6f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.1)
  Log-marginal-likelihood : -216.8160
  Sanity check at best X* : GP mean=8.1694  actual=8.1694  std=0.000010


In [4]:
# Cell 5: PyTorch NN Surrogate (Assignment 16.1 style)
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y.reshape(-1, 1)).ravel().astype(np.float32)

X_t = torch.tensor(X.astype(np.float32))
Y_t = torch.tensor(Y_scaled)

class SurrogateNN(nn.Module):
    def __init__(self, n_in, h1=64, h2=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h1), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(h2, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SurrogateNN(n_dims)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

for epoch in range(1500):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_t), Y_t)
    loss.backward()
    optimizer.step()
    if epoch % 300 == 0:
        print(f'  Epoch {epoch:4d}  MSE = {loss.item():.4f}')

print(f'\n  Final MSE: {loss.item():.4f}')
model.eval()
with torch.no_grad():
    nn_pred_best = model(torch.tensor(best_X.astype(np.float32))).item()
print(f'  NN at best X*: {nn_pred_best:.4f} (scaled)')

  Epoch    0  MSE = 1.0464


  Epoch  300  MSE = 0.2227


  Epoch  600  MSE = 0.0218


  Epoch  900  MSE = 0.0089


  Epoch 1200  MSE = 0.0163



  Final MSE: 0.0145
  NN at best X*: 3.9425 (scaled)


In [5]:
# Cell 6: Dual Acquisition -- NN gradient ascent vs GP UCB
# SVM will pick the final candidate in the next cell.
import torch

trust_radius = 0.05
lo = np.clip(best_X - trust_radius, 0.0, 1.0).astype(np.float32)
hi = np.clip(best_X + trust_radius, 0.0, 1.0).astype(np.float32)
lo_t = torch.tensor(lo)
hi_t = torch.tensor(hi)

# Strategy A: NN gradient ascent
x_opt = torch.tensor(best_X.copy().astype(np.float32), requires_grad=True)
opt_inner = torch.optim.Adam([x_opt], lr=5e-3)
model.eval()
for step in range(800):
    opt_inner.zero_grad()
    (-model(x_opt)).backward()
    opt_inner.step()
    with torch.no_grad():
        x_opt.clamp_(lo_t, hi_t)
grad_candidate = x_opt.detach().numpy().copy()

# Strategy B: GP
np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(30000, n_dims))
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
ucb_scores = gp_mean + 1.0 * gp_std
gp_candidate = X_grid[np.argmax(ucb_scores)]

# NN scores for both (used as fallback if SVM finds both poor)
model.eval()
with torch.no_grad():
    nn_score_grad = model(torch.tensor(grad_candidate.astype(np.float32))).item()
    nn_score_gp   = model(torch.tensor(gp_candidate.astype(np.float32))).item()

# Preliminary winner by NN score (SVM may override)
selected = 'NN gradient ascent' if nn_score_grad >= nn_score_gp else 'GP UCB'
next_x   = grad_candidate if nn_score_grad >= nn_score_gp else gp_candidate
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  Dual Acquisition (preliminary -- SVM filter next)')
print('=' * 72)
grad_str = ', '.join(f'{v:.6f}' for v in grad_candidate)
gp_str   = ', '.join(f'{v:.6f}' for v in gp_candidate)
print(f'  NN gradient candidate : [{grad_str}]  (NN score: {nn_score_grad:.4f})')
print(f'  GP UCB candidate      : [{gp_str}]  (NN score: {nn_score_gp:.4f})')
print(f'  Preliminary winner    : {selected}')
print('=' * 72)

  Dual Acquisition (preliminary -- SVM filter next)
  NN gradient candidate : [0.083474, 1.000000, 1.000000, 1.000000]  (NN score: 4.8395)
  GP UCB candidate      : [0.154959, 0.906482, 0.957872, 0.906714]  (NN score: 2.9193)
  Preliminary winner    : NN gradient ascent


In [6]:
# Cell 7: Input Gradient Sensitivity
import torch

x_sens = torch.tensor(best_X.copy().astype(np.float32), requires_grad=True)
model.eval()
model(x_sens).backward()
grads = x_sens.grad.abs().numpy()

best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'Input gradient magnitudes at best X* = [{best_x_str}]:')
print()
max_g = grads.max() if grads.max() > 0 else 1.0
for i, g in enumerate(grads):
    bar = '#' * max(1, int(g * 40 / max_g))
    print(f'  x{i+1}: |grad| = {g:.4f}  {bar}')
print(f'\n  Most sensitive: x{int(np.argmax(grads))+1}')

Input gradient magnitudes at best X* = [0.133474, 0.977830, 0.979582, 0.970578]:

  x1: |grad| = 11.9688  ########################################
  x2: |grad| = 1.5960  #####
  x3: |grad| = 7.3291  ########################
  x4: |grad| = 4.5323  ###############

  Most sensitive: x1


In [7]:
# Cell 8: SVM Active Filter
# Scores both candidates and picks the one with higher P(good).
# If both flagged poor (P < 0.3), falls back to NN-score winner.
from sklearn.svm import SVC

threshold = np.percentile(Y, 70)
labels = (Y >= threshold).astype(int)

print(f'  SVM threshold (70th pct): {threshold:.6e}')
print(f'  Good samples: {labels.sum()} / {len(labels)}')
print()

if labels.sum() >= 2 and (labels == 0).sum() >= 2:
    svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_clf.fit(X, labels)

    prob_grad = svm_clf.predict_proba(grad_candidate.reshape(1, -1))[0, 1]
    prob_gp   = svm_clf.predict_proba(gp_candidate.reshape(1, -1))[0, 1]

    print(f'  P(good) -- NN gradient candidate : {prob_grad:.3f}')
    print(f'  P(good) -- GP candidate          : {prob_gp:.3f}')
    print()

    if max(prob_grad, prob_gp) < 0.3:
        svm_selected = selected + ' (SVM fallback -- both poor, keeping NN winner)'
        print(f'  Both candidates flagged poor. Keeping NN winner: {selected}')
    elif prob_grad >= prob_gp:
        next_x = grad_candidate.copy()
        svm_selected = 'NN gradient ascent (SVM endorsed)'
        portal_string = '-'.join([f'{v:.6f}' for v in next_x])
        print(f'  SVM selects: NN gradient ascent (P={prob_grad:.3f})')
    else:
        next_x = gp_candidate.copy()
        svm_selected = 'GP candidate (SVM endorsed)'
        portal_string = '-'.join([f'{v:.6f}' for v in next_x])
        print(f'  SVM selects: GP candidate (P={prob_gp:.3f})')
else:
    svm_selected = selected + ' (insufficient SVM data)'
    print('  Insufficient class balance for SVM -- keeping NN winner.')

next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'\n  FINAL query : [{next_str}]')
print(f'  Portal      : >>> {portal_string} <<<')

  SVM threshold (70th pct): 2.681142e+02
  Good samples: 7 / 24

  P(good) -- NN gradient candidate : 0.898
  P(good) -- GP candidate          : 0.921

  SVM selects: GP candidate (P=0.921)

  FINAL query : [0.154959, 0.906482, 0.957872, 0.906714]
  Portal      : >>> 0.154959-0.906482-0.957872-0.906714 <<<


In [8]:
print('=' * 72)
print('  SUMMARY -- Week 5 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 5 -- Chemical Yield Optimisation (4D)')
print('  Objective       : Maximise')
print(f'  NN architecture : MLP [{n_dims} -> 64 -> 64 -> 1], Adam lr=1e-3, 1500 epochs')
print(f'  Acquisition     : NN gradient ascent vs GP, SVM active filter')
print(f'  SVM decision    : {svm_selected}')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6e}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 5 Bayesian Optimisation
  Function        : 5 -- Chemical Yield Optimisation (4D)
  Objective       : Maximise
  NN architecture : MLP [4 -> 64 -> 64 -> 1], Adam lr=1e-3, 1500 epochs
  Acquisition     : NN gradient ascent vs GP, SVM active filter
  SVM decision    : GP candidate (SVM endorsed)

  Best X*         : [0.133474, 0.977830, 0.979582, 0.970578]
  Best Y*         : 3.531055e+03

  Next query      : [0.154959, 0.906482, 0.957872, 0.906714]

  Portal submission string:
  >>> 0.154959-0.906482-0.957872-0.906714 <<<


## Week 5 -- Reflection

**Function**: F5 -- Chemical Yield Optimisation (4D)

### Strategy note
W4 gave 3531. Tight exploit near W4 with NN+GP dual. SVM active filter.

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 6
*(fill in after reflecting on result)*